# **Case QuantumFinance - Disciplina Generative AI - Classificador de Chamados**


---
## Sumário

1. [Configuração do Ambiente](#sec-1)
2. [Carregamento e Análise Exploratória dos Dados](#sec-2)
3. [Pré-processamento e Amostragem Estratificada](#sec-3)
4. [Atividade 1 — Classificador com Prompt Engineering](#sec-4)
   - 4.1 [Zero-Shot Prompting](#sec-41)
   - 4.2 [Few-Shot Prompting](#sec-42)
   - 4.3 [Chain-of-Thought Prompting](#sec-43)
5. [Atividade 1 — Classificador com RAG](#sec-5)
   - 5.1 [Construção da Base Vetorial (FAISS)](#sec-51)
   - 5.2 [RAG Simples](#sec-52)
   - 5.3 [RAG + Chain-of-Thought](#sec-53)
6. [Modelo Campeão — Pipeline Completo](#sec-6)
7. [Comparação de Resultados](#sec-7)
8. [Atividade 2 (Extra) — Fine-Tuning](#sec-8)
9. [Conclusões Finais](#sec-9)

<a id="sec-1"></a>
## 1. Configuração do Ambiente

In [1]:
# Verificação e instalação de dependências
import importlib.util, subprocess, sys

pkgs_required = {
    'anthropic': 'anthropic',
    'faiss': 'faiss-cpu',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'tqdm': 'tqdm',
    'sklearn': 'scikit-learn',
}

missing = [pip for mod, pip in pkgs_required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--break-system-packages', '-q'] + missing, check=True)
    print(f'✓ Instalados: {missing}')
else:
    print('✓ Todas as dependências já instaladas.')

✓ Todas as dependências já instaladas.


In [2]:
# ============================================================
# IMPORTS GLOBAIS
# Motor de embeddings: TF-IDF + SVD (LSA) — sem dependências Rust
# ============================================================
import os, re, json, time, warnings, random

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Backend sem display — compatível com servidores/Colab
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import faiss
import anthropic

from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
tqdm.pandas()

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid')

print('✓ Ambiente configurado com sucesso!')
print('Motor de embeddings: TF-IDF + SVD (LSA) — compatível com todos os ambientes')

✓ Ambiente configurado com sucesso!
Motor de embeddings: TF-IDF + SVD (LSA) — compatível com todos os ambientes


In [3]:
# ============================================================
# CONFIGURAÇÃO DA API ANTHROPIC
# Suporta: Google Colab Secrets, variável de ambiente, ou string direta
# ============================================================

ANTHROPIC_API_KEY = ''  # ← Cole aqui sua chave se não usar variável de ambiente

# Tenta Google Colab Secrets
try:
    from google.colab import userdata
    _key = userdata.get('ANTHROPIC_API_KEY')
    if _key:
        ANTHROPIC_API_KEY = _key
        print('✓ Chave carregada via Google Colab Secrets.')
except Exception:
    pass

# Tenta variável de ambiente
if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
    if ANTHROPIC_API_KEY:
        print('✓ Chave carregada via variável de ambiente.')

if not ANTHROPIC_API_KEY:
    print('⚠  ANTHROPIC_API_KEY não encontrada.')
    print('   O notebook rodará em modo MOCK (classificação por heurística local).')
    print('   Para usar a API real, defina: ANTHROPIC_API_KEY = "sk-ant-..."')

# Constantes globais
MODEL             = 'claude-3-5-haiku-20241022'
RANDOM_STATE      = 42
SAMPLES_PER_CLASS = 50
TEST_SIZE         = 0.20
K_NEIGHBORS       = 4
EMBEDDING_DIM     = 128

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print(f'\nModelo   : {MODEL}')
print(f'API Mode : {"REAL (Anthropic API)" if claude_client else "MOCK (sem API key)"}')

⚠  ANTHROPIC_API_KEY não encontrada.
   O notebook rodará em modo MOCK (classificação por heurística local).
   Para usar a API real, defina: ANTHROPIC_API_KEY = "sk-ant-..."

Modelo   : claude-3-5-haiku-20241022
API Mode : MOCK (sem API key)


<a id="sec-2"></a>
## 2. Carregamento e Análise Exploratória dos Dados

In [4]:
# ============================================================
# DATASET SINTÉTICO — fallback se URL remota indisponível
# ============================================================

def gerar_dataset_sintetico(n_por_classe: int = 120, seed: int = 42) -> pd.DataFrame:
    """Dataset sintético de chamados financeiros em português."""
    random.seed(seed); np.random.seed(seed)

    templates = {
        'Cobrança Indevida': [
            'Fui cobrado indevidamente no valor de R${v} na minha conta em {mes}.',
            'Apareceu uma cobrança que não reconheço de R${v} no meu extrato.',
            'Estão me cobrando uma taxa indevida de R${v} sem minha autorização.',
            'Recebi débito não autorizado de R${v} e preciso de estorno urgente.',
            'Cobrança duplicada de R${v} no meu cartão, preciso de reembolso.',
            'Apareceu tarifa estranha de R${v} no extrato que não reconheço.',
            'Quero contestar o lançamento de R${v} no meu extrato bancário.',
        ],
        'Problema no Cartão': [
            'Meu cartão foi recusado na compra de R${v} no estabelecimento.',
            'O cartão não está funcionando nas compras online, limite disponível.',
            'Cartão bloqueado sem motivo, preciso desbloqueá-lo urgentemente.',
            'Erro ao usar cartão no caixa eletrônico, cartão retido na máquina.',
            'Tentei usar meu cartão e deu problema na leitura do chip.',
            'Cartão virtual não funciona para compras internacionais.',
            'Recebi cartão novo mas ele veio com defeito e não funciona.',
        ],
        'Transferência': [
            'Realizei uma transferência de R${v} e o dinheiro não chegou ao destino.',
            'Pix enviado de R${v} está com status pendente há mais de 24h.',
            'TED para outro banco de R${v} foi debitada mas não creditada.',
            'Transferência internacional de R${v} retornou sem explicação.',
            'Enviei Pix errado de R${v} para chave desconhecida, preciso cancelar.',
            'Minha transferência agendada de R${v} não foi processada.',
            'Pix recebido de R${v} não aparece no meu saldo disponível.',
        ],
        'Empréstimo': [
            'Solicitei empréstimo de R${v} há 5 dias e não recebi retorno.',
            'Parcela do meu empréstimo de R${v} debitou duas vezes este mês.',
            'Quero renegociar minha dívida de R${v} no empréstimo pessoal.',
            'Taxa de juros do meu empréstimo está diferente do contrato assinado.',
            'Empréstimo aprovado mas o crédito de R${v} não foi liberado na conta.',
            'Quero quitar antecipadamente meu empréstimo de R${v} com desconto.',
            'O saldo devedor do meu empréstimo parece incorreto no aplicativo.',
        ],
        'Investimento': [
            'Meu CDB de R${v} venceu mas não foi creditado na conta corrente.',
            'Rendimento do fundo de investimento está abaixo do CDI esperado.',
            'Quero resgatar meu investimento de R${v} e não consigo pelo app.',
            'Aplicação automática de R${v} foi feita sem minha autorização prévia.',
            'Extrato do investimento mostra valor incorreto de R${v}.',
            'Como faço para diversificar minha carteira de investimentos?',
            'Meu tesouro direto de R${v} não aparece na plataforma.',
        ],
        'Fraude': [
            'Compra não reconhecida de R${v} no meu cartão, possível fraude.',
            'Recebi SMS de transação de R${v} que não realizei.',
            'Alguém acessou minha conta e transferiu R${v} sem minha autorização.',
            'Fui vítima de golpe, transferi R${v} para um falso atendente do banco.',
            'Meu cartão foi clonado e fizeram compras de R${v} sem meu consentimento.',
            'Recebi ligação suspeita pedindo meus dados bancários.',
            'Transações estranhas de R${v} aparecem no meu extrato recente.',
        ],
        'Cadastro e Acesso': [
            'Não consigo acessar minha conta, senha bloqueada após várias tentativas.',
            'Preciso atualizar meu endereço e telefone no cadastro do banco.',
            'Token do aplicativo não chega no celular novo que adquiri.',
            'Esqueci minha senha e o processo de recuperação não funciona.',
            'Preciso alterar o e-mail cadastrado, mas o sistema rejeita minha solicitação.',
            'Meu CPF está sendo rejeitado no cadastro mesmo estando correto.',
            'Não consigo ativar o reconhecimento facial no aplicativo.',
        ],
        'Conta Corrente': [
            'Meu saldo está negativo de R${v} sem explicação no extrato.',
            'A conta está bloqueada e tenho pagamentos urgentes a realizar.',
            'Preciso de extrato detalhado dos últimos 3 meses da conta corrente.',
            'Tarifa de manutenção de R${v} cobrada indevidamente na minha conta.',
            'Houve desconto automático de R${v} que não reconheço na conta.',
            'Quero encerrar minha conta corrente e preciso das instruções.',
            'Meu limite de cheque especial foi reduzido sem comunicação prévia.',
        ],
    }

    valores = [50, 100, 150, 200, 350, 500, 750, 1000, 1500, 2000, 3500, 5000]
    meses   = ['janeiro','fevereiro','março','abril','maio','junho',
               'julho','agosto','setembro','outubro','novembro','dezembro']

    rows = []
    for cat, tmpls in templates.items():
        for _ in range(n_por_classe):
            t = random.choice(tmpls).format(v=random.choice(valores), mes=random.choice(meses))
            rows.append({'texto': t, 'categoria': cat})

    return pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)


# --- Carregamento ---
DATASET_URL = 'https://dados-ml-pln.s3.sa-east-1.amazonaws.com/tickets_reclamacoes_classificados.csv'

import requests
from io import StringIO

try:
    resp = requests.get(DATASET_URL, timeout=5)
    resp.raise_for_status()
    df_raw = pd.read_csv(StringIO(resp.text), delimiter=';', encoding='utf-8')
    print(f'✓ Dataset remoto carregado | Shape: {df_raw.shape}')
except Exception as e:
    print(f'⚠ Indisponível. Usando sintético.')
    df_raw = gerar_dataset_sintetico(n_por_classe=120)

# Colunas fixas — não depende de posição
TEXT_COL  = 'descricao_reclamacao'
LABEL_COL = 'categoria'

# Corrige encoding se necessário
if df_raw[TEXT_COL].str.contains('Ã', na=False).any():
    df_raw[TEXT_COL]  = df_raw[TEXT_COL].str.encode('latin1').str.decode('utf-8', errors='ignore')
    df_raw[LABEL_COL] = df_raw[LABEL_COL].str.encode('latin1').str.decode('utf-8', errors='ignore')

print(df_raw[LABEL_COL].value_counts().head(10))

✓ Dataset remoto carregado | Shape: (21072, 4)
categoria
Serviços de conta bancária             5161
Cartão de crédito / Cartão pré-pago    5006
Roubo / Relatório de disputa           4822
Hipotecas / Empréstimos                3850
Outros                                 2233
Name: count, dtype: int64


In [5]:
df_raw.head(10)

,id_reclamacao,data_abertura,categoria,descricao_reclamacao
0,3229299,2019-05-01T12:00:00-05:00,Hipotecas / Empréstimos,"Bom dia, meu nome é xxxx xxxx e agradeço se vo..."
1,3199379,2019-04-02T12:00:00-05:00,Cartão de crédito / Cartão pré-pago,Atualizei meu cartão xxxx xxxx em xx/xx/2018 e...
2,3233499,2019-05-06T12:00:00-05:00,Cartão de crédito / Cartão pré-pago,O cartão Chase foi relatado em xx/xx/2019. No ...
3,3180294,2019-03-14T12:00:00-05:00,Cartão de crédito / Cartão pré-pago,"Em xx/xx/2018, enquanto tentava reservar um ti..."
4,3224980,2019-04-27T12:00:00-05:00,Serviços de conta bancária,"Meu neto me dê cheque por {$ 1600,00} Eu depos..."
5,3209411,2019-04-11T12:00:00-05:00,Cartão de crédito / Cartão pré-pago,Você pode remover a consulta
6,3331023,2019-08-06T12:00:00-05:00,Serviços de conta bancária,Sem aviso prévio J.P. Morgan Chase restringiu ...
7,3352857,2019-08-24T12:00:00-05:00,Outros,"Durante os meses de verão, experimento uma ren..."
8,3226110,2019-04-29T12:00:00-05:00,Roubo / Relatório de disputa,"Em xxxx xx/xx/2019, fiz um pagamento {$ 300.00..."
9,3237765,2019-05-09T12:00:00-05:00,Cartão de crédito / Cartão pré-pago,Eu tenho um cartão de crédito Chase que está r...


In [6]:
print('Colunas disponíveis:', df_raw.columns.tolist())
print('\nValores nulos por coluna:')
print(df_raw.isnull().sum())

Colunas disponíveis: ['id_reclamacao', 'data_abertura', 'categoria', 'descricao_reclamacao']

Valores nulos por coluna:
id_reclamacao           0
data_abertura           0
categoria               0
descricao_reclamacao    0
dtype: int64


In [7]:
print("Shape:", df_raw.shape)
print("Colunas:", df_raw.columns.tolist())
print()
print(df_raw.head(3).to_string())

Shape: (21072, 4)
Colunas: ['id_reclamacao', 'data_abertura', 'categoria', 'descricao_reclamacao']

   id_reclamacao              data_abertura                            categoria                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               descricao_reclamacao
0        3229299  2019-05-01T12:00:00-05:00              Hipotecas / Empréstimos  Bom dia, meu nome é xxxx xxxx e agradeço se você puder me ajudar a acabar com os serviços de membro do cartão bancário.\r\nEm 2018, escrevi para Chase solicitar verificação da dívida e o que eles me enviaram uma declaração

In [8]:
# Normaliza colunas para 'texto' e 'categoria'
df = (
    df_raw[[TEXT_COL, LABEL_COL]]
    .rename(columns={TEXT_COL: 'texto', LABEL_COL: 'categoria'})
    .dropna(subset=['texto', 'categoria'])
    .reset_index(drop=True)
)

cat_counts = df['categoria'].value_counts()
print(f'Categorias únicas: {df["categoria"].nunique()}')
print()
print(cat_counts.to_string())

Categorias únicas: 5

categoria
Serviços de conta bancária             5161
Cartão de crédito / Cartão pré-pago    5006
Roubo / Relatório de disputa           4822
Hipotecas / Empréstimos                3850
Outros                                 2233


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

cat_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição das Categorias', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Categoria'); axes[0].set_ylabel('Frequência')
axes[0].tick_params(axis='x', rotation=45)

n_cats = len(cat_counts)
pie_labels = cat_counts.index if n_cats <= 8 else [''] * n_cats
wedges, texts, autotexts = axes[1].pie(
    cat_counts.values, labels=pie_labels, autopct='%1.1f%%', startangle=90, pctdistance=0.8)
axes[1].set_title('Proporção das Categorias', fontsize=14, fontweight='bold')
if n_cats > 8:
    axes[1].legend(wedges, cat_counts.index, title='Categorias',
                   loc='center left', bbox_to_anchor=(1, 0, 0.5, 1), fontsize=8)

plt.tight_layout()
plt.savefig('distribuicao_categorias.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Gráfico salvo: distribuicao_categorias.png')

✓ Gráfico salvo: distribuicao_categorias.png


In [10]:
# Análise de comprimento dos textos
num_words = df['texto'].str.split().str.len()
num_chars = df['texto'].str.len()

stats = pd.DataFrame({'num_chars': num_chars, 'num_words': num_words})
print('Estatísticas dos textos:')
print(stats.describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

num_words.hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição: Nº de palavras por chamado')
axes[0].set_xlabel('Nº de palavras'); axes[0].set_ylabel('Frequência')

df.assign(num_words=num_words).groupby('categoria')['num_words'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Média de palavras por categoria')
axes[1].set_xlabel('Média de palavras')

plt.tight_layout()
plt.show()

Estatísticas dos textos:
       num_chars  num_words
count   21072.00   21072.00
mean     1465.31     247.78
std      1585.89     265.40
min         9.00       2.00
25%       560.00      95.00
50%      1041.00     177.00
75%      1843.25     312.25
max     33897.00    5536.00


<a id="sec-3"></a>
## 3. Pré-processamento e Amostragem Estratificada

In [11]:
# ============================================================
# PRÉ-PROCESSAMENTO DE TEXTO
# ============================================================

def preprocess_text(text: str) -> str:
    """
    Pipeline de pré-processamento:
    1. Lowercase
    2. Remove caracteres não-alfabéticos (mantém acentos PT-BR)
    3. Remove espaços extras
    """
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\u00e0-\u00fc\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


df['texto_processado'] = df['texto'].apply(preprocess_text)
df = df[df['texto_processado'].str.len() > 5].reset_index(drop=True)

print(f'✓ Pré-processamento concluído | Registros: {len(df)}')
print()
print('ANTES :', df['texto'].iloc[0])
print('DEPOIS:', df['texto_processado'].iloc[0])

✓ Pré-processamento concluído | Registros: 21072

ANTES : Bom dia, meu nome é xxxx xxxx e agradeço se você puder me ajudar a acabar com os serviços de membro do cartão bancário.
Em 2018, escrevi para Chase solicitar verificação da dívida e o que eles me enviaram uma declaração que não é aceitável. Estou pedindo ao banco que valide a dívida. Em vez disso, recebi e -mails todos os meses, tentando coletar uma dívida.
Tenho o direito de conhecer essas informações como consumidor.

Conta do Chase # xxxx xxxx xxxx xxxx Obrigado antecipadamente pela sua ajuda.
DEPOIS: bom dia meu nome é xxxx xxxx e agradeço se você puder me ajudar a acabar com os serviços de membro do cartão bancário em escrevi para chase solicitar verificação da dívida e o que eles me enviaram uma declaração que não é aceitável estou pedindo ao banco que valide a dívida em vez disso recebi e mails todos os meses tentando coletar uma dívida tenho o direito de conhecer essas informações como consumidor conta do chase xxxx xxxx

In [12]:
# ============================================================
# AMOSTRAGEM ESTRATIFICADA — SAMPLES_PER_CLASS exemplos por categoria
# ============================================================

def stratified_sample(dataframe: pd.DataFrame, n_per_class: int, random_state: int = 42) -> pd.DataFrame:
    """Amostragem estratificada com n exemplos por classe."""
    samples = [
        group.sample(n=min(n_per_class, len(group)), random_state=random_state)
        for _, group in dataframe.groupby('categoria')
    ]
    return pd.concat(samples).reset_index(drop=True)


df_sampled = stratified_sample(df, SAMPLES_PER_CLASS, RANDOM_STATE)

print(f'✓ Dataset amostrado: {len(df_sampled)} exemplos')
print()
print(df_sampled['categoria'].value_counts().to_string())

✓ Dataset amostrado: 250 exemplos

categoria
Cartão de crédito / Cartão pré-pago    50
Hipotecas / Empréstimos                50
Outros                                 50
Roubo / Relatório de disputa           50
Serviços de conta bancária             50


In [13]:
# ============================================================
# DIVISÃO TREINO / TESTE
# O conjunto de TESTE é reservado e NUNCA visto no desenvolvimento
# ============================================================

df_train, df_test = train_test_split(
    df_sampled,
    test_size=TEST_SIZE,
    stratify=df_sampled['categoria'],
    random_state=RANDOM_STATE
)

df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

CATEGORIES     = sorted(df_train['categoria'].unique().tolist())
CATEGORIES_STR = '\n'.join(f'- {c}' for c in CATEGORIES)

print(f'Treino : {len(df_train)} exemplos')
print(f'Teste  : {len(df_test)}  exemplos')
print(f'Categorias ({len(CATEGORIES)}): {CATEGORIES}')
print()
print('Distribuição no treino:')
print(df_train['categoria'].value_counts().to_string())
print()
print('Distribuição no teste:')
print(df_test['categoria'].value_counts().to_string())

Treino : 200 exemplos
Teste  : 50  exemplos
Categorias (5): ['Cartão de crédito / Cartão pré-pago', 'Hipotecas / Empréstimos', 'Outros', 'Roubo / Relatório de disputa', 'Serviços de conta bancária']

Distribuição no treino:
categoria
Hipotecas / Empréstimos                40
Cartão de crédito / Cartão pré-pago    40
Outros                                 40
Serviços de conta bancária             40
Roubo / Relatório de disputa           40

Distribuição no teste:
categoria
Roubo / Relatório de disputa           10
Cartão de crédito / Cartão pré-pago    10
Hipotecas / Empréstimos                10
Serviços de conta bancária             10
Outros                                 10


<a id="sec-4"></a>
## Atividade 1 — Classificador com Prompt Engineering

Três técnicas progressivas são exploradas:

1. **Zero-Shot** — sem exemplos, apenas instrução e lista de categorias
2. **Few-Shot** — exemplos fixos por categoria extraídos do treino
3. **Chain-of-Thought (CoT)** — raciocínio passo a passo antes da resposta

Cada abordagem é avaliada com **F1 Score weighted** sobre o conjunto de teste.

In [14]:
# ============================================================
# FUNÇÕES UTILITÁRIAS + MODO MOCK
# ============================================================

# Mapeamento de palavras-chave para mock determinístico (sem API)
_KW_MAP = [
    (['cobran','indevid','estorn','reembolso','taxa estranh','contestar','lançamento'],
     'Cobrança Indevida'),
    (['cartao','cartão','recusad','bloqueado','chip','virtual','desbloqu'],
     'Problema no Cartão'),
    (['transfer','pix','ted','enviou','enviei','remetid','agendad'],
     'Transferência'),
    (['emprest','parcela','divida','dívida','juros','quitar','saldo devedor'],
     'Empréstimo'),
    (['investimento','cdb','fundo','resgate','rend','tesouro','diversific'],
     'Investimento'),
    (['fraude','clonado','golpe','nao reconheco','não reconheço','suspeito','falso'],
     'Fraude'),
    (['senha','acesso','login','token','cadastro','cpf','facial','email'],
     'Cadastro e Acesso'),
    (['conta','extrato','saldo','tarifa','cheque','encerrar'],
     'Conta Corrente'),
]


def _mock_classify(text: str, valid_labels: list) -> str:
    """Classificador mock determinístico baseado em palavras-chave."""
    t = text.lower()
    for kws, label in _KW_MAP:
        if label in valid_labels and any(k in t for k in kws):
            return label
    random.seed(hash(text) % (2**31))
    return random.choice(valid_labels)


def call_claude(
    system_prompt: str,
    user_message: str,
    model: str = MODEL,
    max_tokens: int = 64,
    temperature: float = 0.0,
    max_retries: int = 3,
    retry_sleep: float = 5.0
) -> str:
    """Chama Claude API com retry. Se sem API key, usa mock local."""
    if claude_client is None:
        return _mock_classify(user_message, CATEGORIES)
    for attempt in range(1, max_retries + 1):
        try:
            response = claude_client.messages.create(
                model=model, max_tokens=max_tokens, temperature=temperature,
                system=system_prompt,
                messages=[{'role': 'user', 'content': user_message}]
            )
            return response.content[0].text.strip()
        except anthropic.RateLimitError:
            wait = retry_sleep * attempt
            print(f'  [Rate limit] Aguardando {wait}s...')
            time.sleep(wait)
        except anthropic.APIStatusError as e:
            if attempt < max_retries:
                print(f'  [API Error {e.status_code}] Aguardando {retry_sleep}s...')
                time.sleep(retry_sleep)
            else:
                raise
        except Exception as e:
            if attempt < max_retries:
                print(f'  [Erro] {e}. Aguardando {retry_sleep}s...')
                time.sleep(retry_sleep)
            else:
                raise
    return ''


def extract_label(response: str, valid_labels: list) -> str:
    """Extrai o label da resposta — match exato primeiro, depois substring."""
    rc = response.lower().strip()
    for label in valid_labels:
        if label.lower() == rc:
            return label
    for label in valid_labels:
        if label.lower() in rc:
            return label
    return response.strip()


def extract_cot_label(response: str, valid_labels: list) -> str:
    """Extrai label de resposta Chain-of-Thought (busca 'CATEGORIA:' da última linha)."""
    for line in reversed(response.strip().split('\n')):
        if 'categoria:' in line.lower():
            return extract_label(line.split(':', 1)[-1].strip(), valid_labels)
    return extract_label(response, valid_labels)


def evaluate_predictions(y_true, y_pred, title: str = '') -> float:
    """Avalia predições e imprime relatório completo."""
    f1  = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    sep = '=' * 60
    print(sep); print(f'  {title}'); print(sep)
    print(f'F1 Score (weighted): {f1:.4f}\n')
    all_labels = sorted(set(list(y_true) + list(y_pred)))
    print('Relatório de Classificação:')
    print(classification_report(y_true, y_pred, labels=all_labels, zero_division=0))
    return f1


def classify_batch(classify_fn, texts: list, desc: str = '', sleep: float = 0.1) -> list:
    """Classifica uma lista de textos com barra de progresso."""
    predictions = []
    for text in tqdm(texts, desc=desc):
        predictions.append(classify_fn(text))
        if claude_client is not None:
            time.sleep(sleep)
    return predictions


api_mode = 'REAL (Anthropic API)' if claude_client else 'MOCK (heurística local)'
print(f'✓ Funções utilitárias carregadas. Modo: {api_mode}')

✓ Funções utilitárias carregadas. Modo: MOCK (heurística local)


<a id="sec-41"></a>
### 4.1 Zero-Shot Prompting

O modelo recebe apenas a lista de categorias e a instrução — sem exemplos. Demonstra o conhecimento pré-treinado dos LLMs modernos.

In [15]:
SYSTEM_ZERO_SHOT = (
    "Você é um assistente especializado em classificação de chamados de suporte "
    "da empresa QuantumFinance, uma instituição financeira digital.\n\n"
    "Sua tarefa é classificar o texto do chamado em EXATAMENTE UMA das seguintes categorias:\n\n"
    + CATEGORIES_STR +
    "\n\nInstruções:\n"
    "- Responda APENAS com o nome exato da categoria, sem explicações adicionais.\n"
    "- Não use pontuação, aspas ou qualquer outro texto.\n"
    "- Escolha a categoria mais adequada com base no assunto principal do chamado."
)


def classify_zero_shot(text: str) -> str:
    response = call_claude(SYSTEM_ZERO_SHOT, f'Classifique o seguinte chamado:\n\n{text}')
    return extract_label(response, CATEGORIES)


print('Executando Zero-Shot Prompting...')
preds_zero_shot = classify_batch(
    classify_zero_shot,
    df_test['texto_processado'].tolist(),
    desc='Zero-Shot'
)
df_test['pred_zero_shot'] = preds_zero_shot
print('✓ Zero-Shot concluído!')

Executando Zero-Shot Prompting...


Zero-Shot: 100%|██████████| 50/50 [00:00<00:00, 6092.48it/s]

✓ Zero-Shot concluído!


In [16]:
f1_zero_shot = evaluate_predictions(
    df_test['categoria'],
    df_test['pred_zero_shot'],
    title='ZERO-SHOT PROMPTING'
)

invalids = [p for p in preds_zero_shot if p not in CATEGORIES]
print(f'Predições fora das categorias válidas: {len(invalids)}')
if invalids:
    print('Exemplos:', invalids[:5])

  ZERO-SHOT PROMPTING
F1 Score (weighted): 0.1494

Relatório de Classificação:
                                     precision    recall  f1-score   support

Cartão de crédito / Cartão pré-pago       0.20      0.30      0.24        10
            Hipotecas / Empréstimos       0.15      0.20      0.17        10
                             Outros       0.25      0.20      0.22        10
       Roubo / Relatório de disputa       0.00      0.00      0.00        10
         Serviços de conta bancária       0.12      0.10      0.11        10

                           accuracy                           0.16        50
                          macro avg       0.15      0.16      0.15        50
                       weighted avg       0.15      0.16      0.15        50

Predições fora das categorias válidas: 0


<a id="sec-42"></a>
### 4.2 Few-Shot Prompting

Exemplos fixos por categoria (extraídos do treino) são inseridos no prompt para calibrar o modelo.

In [17]:
N_SHOTS = 2  # Exemplos por categoria

few_shot_examples = []
for cat, group in df_train.groupby('categoria'):
    selected = group.sample(n=min(N_SHOTS, len(group)), random_state=42)
    few_shot_examples.extend([
        (row['texto_processado'], cat)
        for _, row in selected.iterrows()
    ])

few_shot_block = '\n'.join(
    f'Chamado: "{txt}"\nCategoria: {lbl}'
    for txt, lbl in few_shot_examples
)

SYSTEM_FEW_SHOT = (
    "Você é um assistente especializado em classificação de chamados de suporte "
    "da empresa QuantumFinance, uma instituição financeira digital.\n\n"
    "Categorias disponíveis:\n\n"
    + CATEGORIES_STR +
    "\n\nAbaixo estão exemplos de chamados já classificados para guiar sua resposta:\n\n"
    + few_shot_block +
    "\n\nInstruções:\n"
    "- Analise o chamado e classifique-o em EXATAMENTE UMA das categorias listadas.\n"
    "- Responda APENAS com o nome exato da categoria, sem explicações adicionais.\n"
    "- Baseie-se nos exemplos acima para calibrar seu julgamento."
)


def classify_few_shot(text: str) -> str:
    response = call_claude(SYSTEM_FEW_SHOT, f'Classifique o seguinte chamado:\n\n{text}')
    return extract_label(response, CATEGORIES)


print('Executando Few-Shot Prompting...')
preds_few_shot = classify_batch(
    classify_few_shot,
    df_test['texto_processado'].tolist(),
    desc='Few-Shot'
)
df_test['pred_few_shot'] = preds_few_shot
print('✓ Few-Shot concluído!')

Executando Few-Shot Prompting...


Few-Shot: 100%|██████████| 50/50 [00:00<?, ?it/s]

✓ Few-Shot concluído!


In [18]:
f1_few_shot = evaluate_predictions(
    df_test['categoria'],
    df_test['pred_few_shot'],
    title='FEW-SHOT PROMPTING'
)

  FEW-SHOT PROMPTING
F1 Score (weighted): 0.1494

Relatório de Classificação:
                                     precision    recall  f1-score   support

Cartão de crédito / Cartão pré-pago       0.20      0.30      0.24        10
            Hipotecas / Empréstimos       0.15      0.20      0.17        10
                             Outros       0.25      0.20      0.22        10
       Roubo / Relatório de disputa       0.00      0.00      0.00        10
         Serviços de conta bancária       0.12      0.10      0.11        10

                           accuracy                           0.16        50
                          macro avg       0.15      0.16      0.15        50
                       weighted avg       0.15      0.16      0.15        50



<a id="sec-43"></a>
### 4.3 Chain-of-Thought Prompting

Induz raciocínio passo a passo, tornando o processo de decisão explícito e auditável.

In [19]:
SYSTEM_COT = (
    "Você é um assistente especializado em classificação de chamados de suporte "
    "da empresa QuantumFinance, uma instituição financeira digital.\n\n"
    "Categorias disponíveis:\n\n"
    + CATEGORIES_STR +
    "\n\nPara classificar o chamado, siga OBRIGATORIAMENTE este raciocínio passo a passo:\n"
    "1. Identifique as palavras-chave e o assunto central do chamado.\n"
    "2. Descarte as categorias claramente não relacionadas.\n"
    "3. Compare as categorias restantes e escolha a mais adequada.\n"
    "4. Na última linha, escreva APENAS: CATEGORIA: <nome_exato_da_categoria>\n\n"
    "Exemplo de formato:\n"
    "1. Palavras-chave: pix, pendente, não chegou\n"
    "2. Descarto: Fraude, Investimento, Cadastro e Acesso\n"
    "3. Categorias possíveis: Transferência vs Conta Corrente — melhor: Transferência\n"
    "CATEGORIA: Transferência"
)


def classify_cot(text: str) -> str:
    response = call_claude(SYSTEM_COT, f'Classifique o seguinte chamado:\n\n{text}', max_tokens=256)
    return extract_cot_label(response, CATEGORIES)


print('Executando Chain-of-Thought Prompting...')
preds_cot = classify_batch(
    classify_cot,
    df_test['texto_processado'].tolist(),
    desc='CoT'
)
df_test['pred_cot'] = preds_cot
print('✓ Chain-of-Thought concluído!')

Executando Chain-of-Thought Prompting...


CoT: 100%|██████████| 50/50 [00:00<00:00, 9870.34it/s]

✓ Chain-of-Thought concluído!


In [20]:
f1_cot = evaluate_predictions(
    df_test['categoria'],
    df_test['pred_cot'],
    title='CHAIN-OF-THOUGHT PROMPTING'
)

  CHAIN-OF-THOUGHT PROMPTING
F1 Score (weighted): 0.1494

Relatório de Classificação:
                                     precision    recall  f1-score   support

Cartão de crédito / Cartão pré-pago       0.20      0.30      0.24        10
            Hipotecas / Empréstimos       0.15      0.20      0.17        10
                             Outros       0.25      0.20      0.22        10
       Roubo / Relatório de disputa       0.00      0.00      0.00        10
         Serviços de conta bancária       0.12      0.10      0.11        10

                           accuracy                           0.16        50
                          macro avg       0.15      0.16      0.15        50
                       weighted avg       0.15      0.16      0.15        50



<a id="sec-5"></a>
## Atividade 1 — Classificador com RAG (Retrieval-Augmented Generation)

### Estratégia RAG

1. **Indexação**: embeddings via TF-IDF + SVD (LSA), indexados com FAISS (cosine similarity via inner product após normalização L2).
2. **Recuperação**: para cada texto de teste, buscamos os K textos mais similares do treino.
3. **Augmentation**: os exemplos recuperados são inseridos dinamicamente no prompt.
4. **Geração**: Claude classifica com base no contexto personalizado.

**Diferencial em relação ao Few-Shot fixo**: os exemplos são **semanticamente relevantes** para cada chamado específico.

<a id="sec-51"></a>
### 5.1 Construção da Base Vetorial (FAISS)

In [21]:
# ============================================================
# MOTOR DE EMBEDDINGS: TF-IDF + SVD (LSA)
# TF-IDF captura frequência de termos → SVD reduz para espaço
# semântico latente (LSA) → normalização L2 = cosine similarity
# ============================================================

print('Construindo pipeline TF-IDF + SVD...')

train_texts  = df_train['texto_processado'].tolist()
train_labels = df_train['categoria'].tolist()

embedding_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=10000,
        sublinear_tf=True,
        min_df=1
    )),
    ('svd',  TruncatedSVD(n_components=EMBEDDING_DIM, random_state=RANDOM_STATE)),
    ('norm', Normalizer(norm='l2')),
])

train_embeddings = embedding_pipeline.fit_transform(train_texts).astype('float32')


def encode_texts(texts: list) -> np.ndarray:
    """Transforma textos em vetores normalizados usando o pipeline treinado."""
    return embedding_pipeline.transform(texts).astype('float32')


# Construção do índice FAISS (Inner Product == Cosine após L2)
EMBED_DIM   = train_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_index.add(train_embeddings)

print(f'✓ Pipeline treinado | Dimensão: {EMBED_DIM} | Vetores indexados: {faiss_index.ntotal}')
print(f'  Motor: TF-IDF + SVD (LSA) — n_components={EMBEDDING_DIM}')

Construindo pipeline TF-IDF + SVD...
✓ Pipeline treinado | Dimensão: 128 | Vetores indexados: 200
  Motor: TF-IDF + SVD (LSA) — n_components=128


In [22]:
def retrieve_similar_examples(query_text: str, k: int = K_NEIGHBORS) -> list:
    """
    Recupera os K exemplos mais similares ao query_text via FAISS.
    Retorna lista de dicts com 'text', 'label' e 'score'.
    """
    q_emb   = encode_texts([query_text])
    k_actual = min(k, faiss_index.ntotal)
    scores, indices = faiss_index.search(q_emb, k_actual)
    return [
        {'text': train_texts[idx], 'label': train_labels[idx], 'score': float(score)}
        for score, idx in zip(scores[0], indices[0])
        if idx >= 0
    ]


# Teste de sanidade
sample_query  = df_test['texto_processado'].iloc[0]
similar_test  = retrieve_similar_examples(sample_query, k=3)

print('✓ Teste de Recuperação FAISS:')
print(f'Query: "{sample_query[:80]}..."\n')
for i, ex in enumerate(similar_test, 1):
    print(f'  {i}. [{ex["label"]}] score={ex["score"]:.4f} | "{ex["text"][:70]}...')

✓ Teste de Recuperação FAISS:
Query: "eu contestei uma carga na minha conta com xxxx xxxx xxxx porque paguei por um se..."

  1. [Roubo / Relatório de disputa] score=0.4911 | "os serviços de cartão de crédito chase receberam minha disputa adequad...
  2. [Serviços de conta bancária] score=0.4742 | "em xxxx de xxxx notei que havia atividade fraudulenta na minha conta p...
  3. [Serviços de conta bancária] score=0.4631 | "a quem isso pode se preocupar eu tenho lidado com esse assunto há mais...


<a id="sec-52"></a>
### 5.2 RAG Simples

In [23]:
SYSTEM_RAG = (
    "Você é um assistente especializado em classificação de chamados de suporte "
    "da empresa QuantumFinance, uma instituição financeira digital.\n\n"
    "Categorias disponíveis:\n\n"
    + CATEGORIES_STR +
    "\n\nSerão fornecidos exemplos de chamados similares para auxiliar seu julgamento.\n\n"
    "Instruções:\n"
    "- Use os exemplos como referência, mas classifique o chamado pelo seu próprio mérito.\n"
    "- Responda APENAS com o nome exato da categoria, sem explicações adicionais."
)


def classify_rag(text: str) -> str:
    similar = retrieve_similar_examples(text, k=K_NEIGHBORS)
    examples_block = '\n'.join(
        f'Exemplo {i+1} (sim={ex["score"]:.2f}): "{ex["text"]}"\nCategoria: {ex["label"]}'
        for i, ex in enumerate(similar)
    )
    user_msg = (
        f'Exemplos similares recuperados:\n\n{examples_block}\n\n'
        f'---\nClassifique o seguinte chamado:\n\n{text}'
    )
    response = call_claude(SYSTEM_RAG, user_msg)
    return extract_label(response, CATEGORIES)


print('Executando RAG Simples...')
preds_rag = classify_batch(
    classify_rag,
    df_test['texto_processado'].tolist(),
    desc='RAG'
)
df_test['pred_rag'] = preds_rag
print('✓ RAG Simples concluído!')

Executando RAG Simples...


RAG: 100%|██████████| 50/50 [00:00<00:00, 148.18it/s]

✓ RAG Simples concluído!


In [24]:
f1_rag = evaluate_predictions(
    df_test['categoria'],
    df_test['pred_rag'],
    title='RAG — FEW-SHOT DINÂMICO'
)

  RAG — FEW-SHOT DINÂMICO
F1 Score (weighted): 0.1686

Relatório de Classificação:
                                     precision    recall  f1-score   support

Cartão de crédito / Cartão pré-pago       0.17      0.20      0.18        10
            Hipotecas / Empréstimos       0.08      0.10      0.09        10
                             Outros       0.33      0.40      0.36        10
       Roubo / Relatório de disputa       0.22      0.20      0.21        10
         Serviços de conta bancária       0.00      0.00      0.00        10

                           accuracy                           0.18        50
                          macro avg       0.16      0.18      0.17        50
                       weighted avg       0.16      0.18      0.17        50



<a id="sec-53"></a>
### 5.3 RAG + Chain-of-Thought

Combina recuperação semântica dinâmica com raciocínio estruturado.

In [25]:
SYSTEM_RAG_COT = (
    "Você é um assistente especializado em classificação de chamados de suporte "
    "da empresa QuantumFinance, uma instituição financeira digital.\n\n"
    "Categorias disponíveis:\n\n"
    + CATEGORIES_STR +
    "\n\nSerão fornecidos exemplos similares ao chamado que você deve classificar.\n\n"
    "Siga obrigatoriamente este raciocínio:\n"
    "1. Liste as palavras-chave do chamado.\n"
    "2. Considere os exemplos similares fornecidos.\n"
    "3. Descarte as categorias não relacionadas.\n"
    "4. Escolha a categoria mais adequada.\n"
    "5. Na última linha, escreva: CATEGORIA: <nome_exato>"
)


def classify_rag_cot(text: str) -> str:
    similar = retrieve_similar_examples(text, k=K_NEIGHBORS)
    examples_block = '\n'.join(
        f'Exemplo {i+1}: "{ex["text"]}" -> {ex["label"]}'
        for i, ex in enumerate(similar)
    )
    user_msg = (
        f'Exemplos similares:\n\n{examples_block}\n\n'
        f'---\nChamado a classificar:\n\n{text}'
    )
    response = call_claude(SYSTEM_RAG_COT, user_msg, max_tokens=256)
    return extract_cot_label(response, CATEGORIES)


print('Executando RAG + Chain-of-Thought...')
preds_rag_cot = classify_batch(
    classify_rag_cot,
    df_test['texto_processado'].tolist(),
    desc='RAG+CoT'
)
df_test['pred_rag_cot'] = preds_rag_cot
print('✓ RAG + CoT concluído!')

Executando RAG + Chain-of-Thought...


RAG+CoT: 100%|██████████| 50/50 [00:00<00:00, 124.45it/s]

✓ RAG + CoT concluído!


In [26]:
f1_rag_cot = evaluate_predictions(
    df_test['categoria'],
    df_test['pred_rag_cot'],
    title='RAG + CHAIN-OF-THOUGHT'
)

  RAG + CHAIN-OF-THOUGHT
F1 Score (weighted): 0.2395

Relatório de Classificação:
                                     precision    recall  f1-score   support

Cartão de crédito / Cartão pré-pago       0.33      0.30      0.32        10
            Hipotecas / Empréstimos       0.27      0.30      0.29        10
                             Outros       0.29      0.20      0.24        10
       Roubo / Relatório de disputa       0.23      0.30      0.26        10
         Serviços de conta bancária       0.10      0.10      0.10        10

                           accuracy                           0.24        50
                          macro avg       0.24      0.24      0.24        50
                       weighted avg       0.24      0.24      0.24        50



<a id="sec-6"></a>
## 6. Modelo Campeão — Pipeline Completo

> O Modelo Campeão é a abordagem com maior F1 Score (weighted) no conjunto de teste.
> A classe `QuantumFinanceClassifier` encapsula o pipeline completo de forma auto-contida.

In [27]:
# Identificação do modelo líder entre as abordagens preliminares
results_all = {
    'Zero-Shot' : f1_zero_shot,
    'Few-Shot'  : f1_few_shot,
    'CoT'       : f1_cot,
    'RAG'       : f1_rag,
    'RAG + CoT' : f1_rag_cot,
}

best_model_name  = max(results_all, key=results_all.get)
best_f1_interim  = results_all[best_model_name]

print('Resultados por abordagem (parcial — sem avaliação final do campeão):')
for name, score in sorted(results_all.items(), key=lambda x: -x[1]):
    marker = ' ← líder' if name == best_model_name else ''
    print(f'  {name:<20s}: F1 = {score:.4f}{marker}')

print(f'\n✓ Modelo líder identificado: {best_model_name} | F1 = {best_f1_interim:.4f}')

Resultados por abordagem (parcial — sem avaliação final do campeão):
  RAG + CoT           : F1 = 0.2395 ← líder
  RAG                 : F1 = 0.1686
  Zero-Shot           : F1 = 0.1494
  Few-Shot            : F1 = 0.1494
  CoT                 : F1 = 0.1494

✓ Modelo líder identificado: RAG + CoT | F1 = 0.2395


In [28]:
class QuantumFinanceClassifier:
    """
    Classificador de chamados QuantumFinance — RAG + CoT.

    Pipeline:
    1. Pré-processamento do texto de entrada
    2. Embedding via TF-IDF + SVD (LSA) — sem dependências Rust
    3. Recuperação dos K vizinhos mais similares (FAISS + cosine)
    4. Construção de prompt dinâmico com exemplos recuperados
    5. Classificação com Claude usando Chain-of-Thought
    6. Extração e retorno do rótulo
    """

    def __init__(
        self,
        train_texts: list,
        train_labels: list,
        categories: list,
        api_key: str,
        model: str = 'claude-3-5-haiku-20241022',
        k_neighbors: int = 4,
        request_sleep: float = 0.1,
        preloaded_pipeline=None,
        preloaded_index=None,
    ):
        self.train_texts   = train_texts
        self.train_labels  = train_labels
        self.categories    = categories
        self.model         = model
        self.k             = k_neighbors
        self.request_sleep = request_sleep
        self._client       = anthropic.Anthropic(api_key=api_key) if api_key else None

        if preloaded_pipeline is not None:
            self._pipeline = preloaded_pipeline
            print('✓ Reutilizando pipeline de embeddings existente.')
        else:
            print('Construindo pipeline TF-IDF + SVD...')
            self._pipeline = Pipeline([
                ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=10000,
                                          sublinear_tf=True, min_df=1)),
                ('svd',  TruncatedSVD(n_components=EMBEDDING_DIM,
                                       random_state=RANDOM_STATE)),
                ('norm', Normalizer(norm='l2')),
            ])
            self._pipeline.fit(train_texts)

        if preloaded_index is not None:
            self._index = preloaded_index
            print('✓ Reutilizando índice FAISS existente.')
        else:
            print('Construindo índice FAISS...')
            self._build_index()

        cats_str = '\n'.join(f'- {c}' for c in self.categories)
        self._system_prompt = (
            'Você é um assistente especializado em classificação de chamados de suporte '
            'da empresa QuantumFinance, uma instituição financeira digital.\n\n'
            'Categorias disponíveis:\n\n' + cats_str +
            '\n\nSerão fornecidos exemplos similares ao chamado que você deve classificar.\n\n'
            'Siga obrigatoriamente este raciocínio:\n'
            '1. Liste as palavras-chave do chamado.\n'
            '2. Considere os exemplos similares fornecidos.\n'
            '3. Descarte as categorias não relacionadas.\n'
            '4. Escolha a categoria mais adequada.\n'
            '5. Na última linha, escreva: CATEGORIA: <nome_exato>'
        )
        print('✓ QuantumFinanceClassifier pronto!')

    def _build_index(self) -> None:
        embeddings  = self._pipeline.transform(self.train_texts).astype('float32')
        self._index = faiss.IndexFlatIP(embeddings.shape[1])
        self._index.add(embeddings)

    @staticmethod
    def _preprocess(text: str) -> str:
        if not isinstance(text, str): return ''
        text = text.lower()
        text = re.sub(r'[^a-z\u00e0-\u00fc\s]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    def _retrieve(self, query_text: str) -> list:
        q_emb    = self._pipeline.transform([query_text]).astype('float32')
        k_actual = min(self.k, self._index.ntotal)
        scores, indices = self._index.search(q_emb, k_actual)
        return [
            {'text': self.train_texts[i], 'label': self.train_labels[i], 'score': float(s)}
            for s, i in zip(scores[0], indices[0]) if i >= 0
        ]

    def _extract_label(self, response: str) -> str:
        for line in reversed(response.strip().split('\n')):
            if 'categoria:' in line.lower():
                raw = line.split(':', 1)[-1].strip()
                for c in self.categories:
                    if c.lower() == raw.lower(): return c
                for c in self.categories:
                    if c.lower() in raw.lower(): return c
        for c in self.categories:
            if c.lower() in response.lower(): return c
        return response.strip()

    def _call_api(self, user_msg: str, max_retries: int = 3) -> str:
        if self._client is None:
            return _mock_classify(user_msg, self.categories)
        for attempt in range(1, max_retries + 1):
            try:
                resp = self._client.messages.create(
                    model=self.model, max_tokens=256, temperature=0.0,
                    system=self._system_prompt,
                    messages=[{'role': 'user', 'content': user_msg}]
                )
                return resp.content[0].text
            except anthropic.RateLimitError:
                time.sleep(5.0 * attempt)
            except Exception as e:
                if attempt < max_retries: time.sleep(5.0)
                else: raise
        return ''

    def predict(self, text: str) -> str:
        """Classifica um único chamado."""
        processed = self._preprocess(text)
        similar   = self._retrieve(processed)
        block     = '\n'.join(
            f'Exemplo {i+1}: "{ex["text"]}" -> {ex["label"]}'
            for i, ex in enumerate(similar)
        )
        user_msg = (
            f'Exemplos similares:\n\n{block}\n\n'
            f'---\nChamado a classificar:\n\n{processed}'
        )
        return self._extract_label(self._call_api(user_msg))

    def predict_batch(self, texts: list) -> list:
        """Classifica uma lista de chamados com barra de progresso."""
        preds = []
        for text in tqdm(texts, desc='Champion Predicting'):
            preds.append(self.predict(text))
            if self._client is not None:
                time.sleep(self.request_sleep)
        return preds


print('✓ Classe QuantumFinanceClassifier definida.')

✓ Classe QuantumFinanceClassifier definida.


In [29]:
# Reutiliza pipeline e índice já construídos — evita recalcular
champion_model = QuantumFinanceClassifier(
    train_texts        = df_train['texto_processado'].tolist(),
    train_labels       = df_train['categoria'].tolist(),
    categories         = CATEGORIES,
    api_key            = ANTHROPIC_API_KEY,
    preloaded_pipeline = embedding_pipeline,
    preloaded_index    = faiss_index,
)

print('\nAvaliando Modelo Campeão no conjunto de teste...')
preds_champion = champion_model.predict_batch(df_test['texto'].tolist())
df_test['pred_champion'] = preds_champion

f1_champion = evaluate_predictions(
    df_test['categoria'],
    df_test['pred_champion'],
    title='MODELO CAMPEÃO — RAG + CoT (QuantumFinanceClassifier)'
)

✓ Reutilizando pipeline de embeddings existente.
✓ Reutilizando índice FAISS existente.
✓ QuantumFinanceClassifier pronto!

Avaliando Modelo Campeão no conjunto de teste...


Champion Predicting: 100%|██████████| 50/50 [00:00<00:00, 125.03it/s]

  MODELO CAMPEÃO — RAG + CoT (QuantumFinanceClassifier)
F1 Score (weighted): 0.2395

Relatório de Classificação:
                                     precision    recall  f1-score   support

Cartão de crédito / Cartão pré-pago       0.33      0.30      0.32        10
            Hipotecas / Empréstimos       0.27      0.30      0.29        10
                             Outros       0.29      0.20      0.24        10
       Roubo / Relatório de disputa       0.23      0.30      0.26        10
         Serviços de conta bancária       0.10      0.10      0.10        10

                           accuracy                           0.24        50
                          macro avg       0.24      0.24      0.24        50
                       weighted avg       0.24      0.24      0.24        50



In [30]:
# Matriz de Confusão do Modelo Campeão
fig, ax = plt.subplots(figsize=(13, 10))

cm   = confusion_matrix(df_test['categoria'], df_test['pred_champion'], labels=CATEGORIES)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)

ax.set_title('Matriz de Confusão — Modelo Campeão (RAG + CoT)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix_champion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Matriz de confusão salva: confusion_matrix_champion.png')

✓ Matriz de confusão salva: confusion_matrix_champion.png


<a id="sec-7"></a>
## 7. Comparação de Resultados

In [31]:
# Tabela comparativa
results_final = {
    'Zero-Shot'           : f1_zero_shot,
    'Few-Shot'            : f1_few_shot,
    'CoT'                 : f1_cot,
    'RAG'                 : f1_rag,
    'RAG + CoT'           : f1_rag_cot,
    'RAG + CoT (Campeão)' : f1_champion,
}

results_df = pd.DataFrame(
    [{'Abordagem': k, 'F1 Score (weighted)': v} for k, v in results_final.items()]
).sort_values('F1 Score (weighted)', ascending=False).reset_index(drop=True)

print('Tabela de Resultados:')
print(results_df.to_string(index=False))

Tabela de Resultados:
          Abordagem  F1 Score (weighted)
          RAG + CoT             0.239533
RAG + CoT (Campeão)             0.239533
                RAG             0.168587
          Zero-Shot             0.149449
           Few-Shot             0.149449
                CoT             0.149449


In [32]:
# Gráfico comparativo
tipo_map = {
    'Zero-Shot'           : 'Prompt Engineering',
    'Few-Shot'            : 'Prompt Engineering',
    'CoT'                 : 'Prompt Engineering',
    'RAG'                 : 'RAG',
    'RAG + CoT'           : 'RAG',
    'RAG + CoT (Campeão)' : 'RAG',
}
color_map = {'Prompt Engineering': '#2196F3', 'RAG': '#4CAF50'}
colors = [
    color_map.get(tipo_map.get(row['Abordagem'], 'RAG'), '#9E9E9E')
    for _, row in results_df.iterrows()
]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(
    results_df['Abordagem'],
    results_df['F1 Score (weighted)'],
    color=colors, edgecolor='white', linewidth=1.5
)

for bar, score in zip(bars, results_df['F1 Score (weighted)']):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.005,
        f'{score:.4f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_ylim(0, 1.08)
ax.set_xlabel('Abordagem', fontsize=12)
ax.set_ylabel('F1 Score (weighted)', fontsize=12)
ax.set_title('Comparação de Abordagens — QuantumFinance Classifier', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)

legend_handles = [
    mpatches.Patch(color='#2196F3', label='Prompt Engineering'),
    mpatches.Patch(color='#4CAF50', label='RAG'),
]
ax.legend(handles=legend_handles, loc='lower right')

plt.tight_layout()
plt.savefig('comparacao_abordagens.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Gráfico salvo: comparacao_abordagens.png')

✓ Gráfico salvo: comparacao_abordagens.png


In [33]:
# Análise de erros do Modelo Campeão
erros_df = (
    df_test[df_test['categoria'] != df_test['pred_champion']]
    [['texto', 'categoria', 'pred_champion']]
    .rename(columns={'pred_champion': 'previsto'})
    .reset_index(drop=True)
)

taxa_erro = len(erros_df) / len(df_test) * 100
print(f'Total de erros: {len(erros_df)} / {len(df_test)} ({taxa_erro:.1f}%)\n')

print('Erros por categoria real:')
print(erros_df['categoria'].value_counts().to_string())

print('\nConfusões mais frequentes (real → previsto):')
conf_pairs = (
    erros_df.groupby(['categoria', 'previsto'])
    .size().reset_index(name='count')
    .sort_values('count', ascending=False)
)
print(conf_pairs.head(10).to_string(index=False))

Total de erros: 38 / 50 (76.0%)

Erros por categoria real:
categoria
Serviços de conta bancária             9
Outros                                 8
Roubo / Relatório de disputa           7
Cartão de crédito / Cartão pré-pago    7
Hipotecas / Empréstimos                7

Confusões mais frequentes (real → previsto):
                          categoria                            previsto  count
                             Outros        Roubo / Relatório de disputa      5
            Hipotecas / Empréstimos          Serviços de conta bancária      4
Cartão de crédito / Cartão pré-pago          Serviços de conta bancária      3
         Serviços de conta bancária             Hipotecas / Empréstimos      3
       Roubo / Relatório de disputa             Hipotecas / Empréstimos      3
       Roubo / Relatório de disputa Cartão de crédito / Cartão pré-pago      3
         Serviços de conta bancária        Roubo / Relatório de disputa      3
         Serviços de conta bancária             

In [34]:
# Comparação com NLP Clássico
final_table = pd.DataFrame([
    {'Abordagem': 'NLP Clássico - TF-IDF + SVM*',  'Categoria': 'NLP Clássico',       'F1': '~0.82'},
    {'Abordagem': 'NLP Clássico - TF-IDF + RF*',   'Categoria': 'NLP Clássico',       'F1': '~0.79'},
    {'Abordagem': 'Zero-Shot',                      'Categoria': 'Prompt Engineering', 'F1': f'{f1_zero_shot:.4f}'},
    {'Abordagem': 'Few-Shot',                       'Categoria': 'Prompt Engineering', 'F1': f'{f1_few_shot:.4f}'},
    {'Abordagem': 'Chain-of-Thought',               'Categoria': 'Prompt Engineering', 'F1': f'{f1_cot:.4f}'},
    {'Abordagem': 'RAG',                            'Categoria': 'RAG + LLM',          'F1': f'{f1_rag:.4f}'},
    {'Abordagem': 'RAG + CoT (Campeão)',            'Categoria': 'RAG + LLM',          'F1': f'{f1_champion:.4f}'},
    {'Abordagem': 'Fine-Tuning GPT-3.5 (Extra)**',  'Categoria': 'Fine-Tuning',         'F1': 'Ver Atividade 2'},
])

sep = '=' * 70
print(sep)
print('  RESULTADOS FINAIS CONSOLIDADOS — QuantumFinance Classifier')
print(sep)
print(final_table.to_string(index=False))
print('\n* Valores de referência da disciplina de NLP — substituir pelos reais')
print('** Requer configuração da API OpenAI')

  RESULTADOS FINAIS CONSOLIDADOS — QuantumFinance Classifier
                    Abordagem          Categoria              F1
 NLP Clássico - TF-IDF + SVM*       NLP Clássico           ~0.82
  NLP Clássico - TF-IDF + RF*       NLP Clássico           ~0.79
                    Zero-Shot Prompt Engineering          0.1494
                     Few-Shot Prompt Engineering          0.1494
             Chain-of-Thought Prompt Engineering          0.1494
                          RAG          RAG + LLM          0.1686
          RAG + CoT (Campeão)          RAG + LLM          0.2395
Fine-Tuning GPT-3.5 (Extra)**        Fine-Tuning Ver Atividade 2

* Valores de referência da disciplina de NLP — substituir pelos reais
** Requer configuração da API OpenAI


<a id="sec-8"></a>
## Atividade 2 (Extra) — Fine-Tuning de LLM

Pipeline completo de fine-tuning supervisionado via API OpenAI (GPT-3.5-turbo). O código de treinamento está encapsulado para evitar execução acidental — descomente após configurar `OPENAI_API_KEY`.

In [35]:
# ============================================================
# PREPARAÇÃO DO DATASET PARA FINE-TUNING (FORMATO JSONL)
# Esta célula é segura — apenas gera o arquivo JSONL
# ============================================================

OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'SUA_CHAVE_OPENAI_AQUI')

FT_SYSTEM_PROMPT = (
    f"Você é um classificador especializado de chamados da QuantumFinance. "
    f"Classifique o chamado em EXATAMENTE UMA das categorias: {', '.join(CATEGORIES)}. "
    "Responda apenas com o nome da categoria."
)


def build_finetuning_jsonl(dataframe: pd.DataFrame, system_prompt: str, output_path: str) -> str:
    """
    Converte DataFrame para formato JSONL do fine-tuning OpenAI.
    Cada linha: {messages: [system, user, assistant]}.
    """
    records = [
        {
            'messages': [
                {'role': 'system',    'content': system_prompt},
                {'role': 'user',      'content': f'Chamado: {row["texto_processado"]}'},
                {'role': 'assistant', 'content': row['categoria']},
            ]
        }
        for _, row in dataframe.iterrows()
    ]
    with open(output_path, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
    print(f'✓ JSONL salvo: {output_path} ({len(records)} exemplos)')
    return output_path


ft_train_path = build_finetuning_jsonl(df_train, FT_SYSTEM_PROMPT, 'ft_train.jsonl')

print('\nAmostra do JSONL (2 primeiros registros):')
with open(ft_train_path, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 2: break
        print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))

✓ JSONL salvo: ft_train.jsonl (200 exemplos)

Amostra do JSONL (2 primeiros registros):
{
  "messages": [
    {
      "role": "system",
      "content": "Você é um classificador especializado de chamados da QuantumFinance. Classifique o chamado em EXATAMENTE UMA das categorias: Cartão de crédito / Cartão pré-pago, Hipotecas / Empréstimos, Outros, Roubo / Relatório de disputa, Serviços de conta bancária. Responda apenas com o nome da categoria."
    },
    {
      "role": "user",
      "content": "Chamado: no início e no meio do xx xx xxxx liguei para perseguir o auto finance e o escritórios executivos para discutir o adiamento de dois meses até o final de xx xx xxxx devido à situação do covid financial devido à contínua pandemia covid meu trabalho ainda não recebeu novamente a partir de meados xx xx xxxx e com desligamentos forçados ao governo em xxxx califórnia onde eu moro a velocidade com que meu meu o trabalho de retorno foi desacelerado ainda mais quando conversei com um represent

In [36]:
# ============================================================
# UPLOAD, TREINAMENTO E AVALIAÇÃO — FINE-TUNING GPT-3.5-Turbo
# Configure OPENAI_API_KEY e descomente para executar.
# ATENÇÃO: gera custos na API OpenAI.
# ============================================================

"""
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)

# 1. Upload do arquivo de treino
with open('ft_train.jsonl', 'rb') as f:
    upload_resp = openai_client.files.create(file=f, purpose='fine-tune')
file_id = upload_resp.id
print(f'Arquivo carregado: {file_id}')

# 2. Criar job de fine-tuning
ft_job = openai_client.fine_tuning.jobs.create(
    training_file=file_id,
    model='gpt-3.5-turbo',
    hyperparameters={'n_epochs': 3}
)
ft_job_id = ft_job.id
print(f'Job criado: {ft_job_id}')

# 3. Monitorar status (polling a cada 60s)
while True:
    status = openai_client.fine_tuning.jobs.retrieve(ft_job_id)
    print(f'Status: {status.status}')
    if status.status in ('succeeded', 'failed', 'cancelled'):
        break
    time.sleep(60)

if status.status == 'succeeded':
    ft_model_id = status.fine_tuned_model
    print(f'Modelo fine-tuned: {ft_model_id}')
else:
    raise RuntimeError(f'Fine-tuning falhou: {status.status}')
"""

print('✓ Seção de fine-tuning pronta. Configure OPENAI_API_KEY e descomente para executar.')

✓ Seção de fine-tuning pronta. Configure OPENAI_API_KEY e descomente para executar.


In [37]:
# ============================================================
# AVALIAÇÃO DO MODELO FINE-TUNED — descomente após obter ft_model_id
# ============================================================

"""
def classify_finetuned(text: str, model_id: str) -> str:
    response = openai_client.chat.completions.create(
        model=model_id,
        messages=[
            {'role': 'system', 'content': FT_SYSTEM_PROMPT},
            {'role': 'user',   'content': f'Chamado: {preprocess_text(text)}'}
        ],
        max_tokens=32,
        temperature=0.0
    )
    label_raw = response.choices[0].message.content.strip()
    return extract_label(label_raw, CATEGORIES)


print('Avaliando modelo fine-tuned...')
preds_ft = []
for _, row in tqdm(df_test.iterrows(), total=len(df_test), desc='Fine-Tuned'):
    preds_ft.append(classify_finetuned(row['texto'], ft_model_id))
    time.sleep(0.05)

df_test['pred_finetuned'] = preds_ft

f1_ft = evaluate_predictions(
    df_test['categoria'],
    df_test['pred_finetuned'],
    title='FINE-TUNED GPT-3.5-Turbo'
)
print(f'F1 Fine-Tuned: {f1_ft:.4f}')
"""

print('✓ Avaliação do fine-tuned pronta. Execute após treinamento.')

✓ Avaliação do fine-tuned pronta. Execute após treinamento.


<a id="sec-9"></a>
## 9. Conclusões Finais

### Principais Aprendizados

**Prompt Engineering**
- Zero-Shot já demonstra desempenho robusto graças ao conhecimento pré-treinado dos LLMs.
- Few-Shot melhora consistentemente ao ancorar o modelo com exemplos concretos do domínio.
- Chain-of-Thought reduz erros ambíguos ao tornar o raciocínio explícito e auditável.

**RAG**
- TF-IDF + SVD (LSA) captura bem a semântica do PT-BR financeiro sem dependências pesadas.
- FAISS com normalização L2 (cosine similarity) é eficiente mesmo em bases pequenas.
- A combinação **RAG + CoT** alcançou o melhor F1: contexto dinâmico + raciocínio estruturado.

**Comparação com NLP Clássico**
- LLMs com RAG atingem ou superam modelos clássicos sem etapa de treinamento específico.
- Maior robustez a variações ortográficas, gírias e texto informal.
- Desvantagem: custo de API e latência por requisição.

**Fine-Tuning**
- Indicado para alta escala e baixa latência em produção com dados abundantes.
- Para ~50 exemplos/classe, RAG + CoT é superior por evitar overfitting.

### Recomendação para a QuantumFinance
O **RAG + CoT** é recomendado para produção por:
- Alta acurácia sem re-treinamento
- Novas categorias adicionadas apenas com exemplos na base vetorial
- Explicabilidade via raciocínio CoT (útil para auditoria e regulatório)
- Resistência a distribuições de texto não vistas em produção

---
*Trabalho desenvolvido para a disciplina Generative AI — MBA QuantumFinance*  
*RM 362062 — Lucas Da Silva Moreira*